# 00 — Rebuild Dataset & Lane Mask Cache

**Purpose:** Download/mount BDD100K data, parse poly2d lane annotations,
render binary lane masks to disk, and verify the dataset pipeline loads correctly.

**Outputs (saved to Drive):**
- `bdd100k/lane_masks/train/*.png` — binary lane masks (640×640)
- `bdd100k/lane_masks/val/*.png` — binary lane masks (640×640)

**Prerequisites:** BDD100K images + lane JSON labels accessible on Drive or Colab disk.

In [ ]:
# ── Mount Drive & install deps ──
from google.colab import drive
drive.mount('/content/drive')

!pip install -q yacs tqdm opencv-python-headless

In [ ]:
# ── Clone or symlink the repo ──
import os, sys

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/EcoCAR-Perception-Pipeline-YOLO26-BDD100K/yolop_vehicle_lane'
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── Configuration (DETR-style path resolution) ──
import os
from pathlib import Path
from lib.utils.drive_dataset import (
    ensure_local_dataset_from_drive,
    find_raw_bdd_root,
    find_lane_polygon_jsons,
)

PROJECT_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
DATASET_NAME = 'bdd100k_vehicle5'

# Resolve packaged dataset exactly like the working DETR line:
# every notebook must recover its own usable dataset root.
try:
    DATASET_ROOT = ensure_local_dataset_from_drive(DATASET_NAME, PROJECT_ROOT, require_lane_dir=False)
except FileNotFoundError:
    DATASET_ROOT = f'/content/{DATASET_NAME}'
    os.makedirs(DATASET_ROOT, exist_ok=True)
    print(f'Packaged dataset not found yet, temporary local root: {DATASET_ROOT}')

# Raw BDD assets may still live in the shared EcoCAR area used by the DETR line.
RAW_BDD_ROOT = find_raw_bdd_root(PROJECT_ROOT)
BDD_IMAGES = os.path.join(RAW_BDD_ROOT, 'images', '100k')
BDD_LABELS = os.path.join(RAW_BDD_ROOT, 'labels', '100k')

lane_jsons = find_lane_polygon_jsons(RAW_BDD_ROOT)
BDD_LANE_JSON_TRAIN = lane_jsons['train']
BDD_LANE_JSON_VAL = lane_jsons['val']

# Render masks into the packaged dataset root so later notebooks can reuse them.
LANE_MASK_OUT = os.path.join(DATASET_ROOT, 'masks')

# Keep original BDD geometry by default.
MASK_W, MASK_H = 1280, 720
LINE_THICKNESS = 8

print(f'PROJECT_ROOT:        {PROJECT_ROOT}')
print(f'DATASET_ROOT:        {DATASET_ROOT}')
print(f'RAW_BDD_ROOT:        {RAW_BDD_ROOT}')
print(f'BDD_IMAGES:          {BDD_IMAGES}')
print(f'BDD_LABELS:          {BDD_LABELS}')
print(f'BDD_LANE_JSON_TRAIN: {BDD_LANE_JSON_TRAIN}')
print(f'BDD_LANE_JSON_VAL:   {BDD_LANE_JSON_VAL}')
print(f'LANE_MASK_OUT:       {LANE_MASK_OUT}')


In [ ]:
# ── Debug lane source before rendering ──
from pathlib import Path
from lib.utils.lane_targets import inspect_json_for_lanes

for split, src in [('train', BDD_LANE_JSON_TRAIN), ('val', BDD_LANE_JSON_VAL)]:
    print(f'
=== Debug lane source: {split} ===')
    print('path:', src)
    print('exists:', os.path.exists(src))
    print('is_file:', os.path.isfile(src))
    print('is_dir:', os.path.isdir(src))
    if os.path.isdir(src):
        samples = sorted(str(p) for p in Path(src).glob('*.json'))[:3]
        print('sample json files:', samples)
    print('inspection:')
    for item in inspect_json_for_lanes(src, limit=2):
        print(item)


In [ ]:
# ── Parse lane labels and render masks ──
from pathlib import Path
from lib.utils.lane_render import convert_bdd_lanes_to_masks, print_lane_stats

for split, json_path in [('train', BDD_LANE_JSON_TRAIN), ('val', BDD_LANE_JSON_VAL)]:
    out_dir = Path(LANE_MASK_OUT) / split
    out_dir.mkdir(parents=True, exist_ok=True)

    existing = len(list(out_dir.glob('*.png')))
    print(f'
=== {split} split ===')
    print(f'  Source: {json_path}')
    print(f'  Output: {out_dir}')
    print(f'  Existing masks: {existing}')

    stats = convert_bdd_lanes_to_masks(
        json_path=json_path,
        output_mask_dir=str(out_dir),
        mask_width=MASK_W,
        mask_height=MASK_H,
        img_width=1280,
        img_height=720,
        line_thickness=LINE_THICKNESS,
        overwrite=False,
    )
    print_lane_stats(stats)
    print(f'  Final mask count: {len(list(out_dir.glob("*.png")))}')


In [ ]:
# ── Verify: count masks and spot-check ──
import cv2
import matplotlib.pyplot as plt
import numpy as np

for split in ['train', 'val']:
    mask_dir = Path(LANE_MASK_OUT) / split
    masks = sorted(mask_dir.glob('*.png'))
    print(f'{split}: {len(masks)} lane masks')

# Visualize a few samples
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
val_masks = sorted((Path(LANE_MASK_OUT) / 'val').glob('*.png'))[:4]
for i, mask_path in enumerate(val_masks):
    mask = cv2.imread(str(mask_path), 0)
    img_path = Path(BDD_IMAGES) / 'val' / (mask_path.stem + '.jpg')
    img = cv2.imread(str(img_path))
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[0, i].imshow(img)
        axes[0, i].set_title(mask_path.stem[:20])
    axes[1, i].imshow(mask, cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].axis('off')
plt.suptitle('Sample Images + Lane Masks')
plt.tight_layout()
plt.show()

In [ ]:

# ── Smoke-test dataset pipeline ──
from lib.config import cfg, update_config
from lib.dataset import BddDataset
from torch.utils.data import DataLoader
import torchvision.transforms as T

cfg.defrost()
cfg.DATASET.ROOT = DATASET_ROOT
cfg.DATASET.DATAROOT = BDD_IMAGES
cfg.DATASET.LABELROOT = BDD_LABELS
cfg.DATASET.LANEROOT = LANE_MASK_OUT
cfg.freeze()

val_dataset = BddDataset(cfg, is_train=False, inputsize=640, transform=T.ToTensor())
print(f'Val dataset: {len(val_dataset)} samples')

val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False, num_workers=2,
    collate_fn=val_dataset.collate_fn
)

img, target, paths, shapes = next(iter(val_loader))
det_labels, lane_labels = target
print(f'Image batch shape:  {img.shape}')
print(f'Det labels shape:   {det_labels.shape}')
print(f'Lane labels shape:  {lane_labels.shape}')
print('Dataset pipeline OK!')


In [ ]:
# ── Persist rebuilt dataset back to Drive for later notebooks ──
import shutil
import tarfile
from pathlib import Path

DRIVE_DATASET_DIR = Path(PROJECT_ROOT) / 'datasets' / DATASET_NAME
DRIVE_TAR_PATH = Path(PROJECT_ROOT) / 'datasets' / f'{DATASET_NAME}.tar'
DRIVE_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

local_root = Path(DATASET_ROOT).resolve()
drive_root = DRIVE_DATASET_DIR.resolve()

if local_root != drive_root:
    if DRIVE_DATASET_DIR.exists():
        shutil.rmtree(DRIVE_DATASET_DIR)
    print(f'Copying rebuilt dataset to Drive: {DRIVE_DATASET_DIR}')
    shutil.copytree(DATASET_ROOT, DRIVE_DATASET_DIR, dirs_exist_ok=True)
else:
    print('Dataset root is already on Drive; skipping directory copy.')

if DRIVE_TAR_PATH.exists():
    DRIVE_TAR_PATH.unlink()
print(f'Packing Drive dataset as tar: {DRIVE_TAR_PATH}')
with tarfile.open(DRIVE_TAR_PATH, 'w') as tar:
    tar.add(str(DRIVE_DATASET_DIR), arcname=DATASET_NAME)

print('Saved for downstream notebooks:')
print(f'  Drive dir: {DRIVE_DATASET_DIR}')
print(f'  Drive tar: {DRIVE_TAR_PATH}')
